In [7]:
"""
Hugging Face’s model: microsoft/resnet-50
"""
import time
from datasets import load_dataset
from torchvision import transforms
from transformers import AutoImageProcessor, AutoModelForImageClassification
import torch
from torch.utils.data import DataLoader
from sklearn.metrics import accuracy_score

# Measure total execution time
start_time = time.time()

# Load dataset
ds = load_dataset("mertcobanov/animals")

# Split dataset into training and testing sets
ds = ds['train'].train_test_split(test_size=0.2, seed=42)
train_data = ds['train']
test_data = ds['test']

# Define the image transformation pipeline
transform = transforms.Compose([
    transforms.Resize((224, 224)),  # Resize images
    transforms.ToTensor(),          # Convert to tensor
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])  # Normalize
])

# Data preprocessing function
def preprocess(batch):
    images = []
    labels = []
    for item in batch:
        if 'image' in item and 'label' in item:
            image = transform(item['image'])  # Apply transformation
            images.append(image)
            labels.append(item['label'])
    return torch.stack(images), torch.tensor(labels)

# Preprocess test data
test_images, test_labels = preprocess(test_data)

# Create DataLoader for testing
test_loader = DataLoader(list(zip(test_images, test_labels)), batch_size=32)

# Load pretrained ResNet-50 model
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = AutoModelForImageClassification.from_pretrained("microsoft/resnet-50")
model.to(device)
model.eval()  # Set model to evaluation mode

# Evaluation function
def evaluate(loader):
    all_preds = []
    all_labels = []
    with torch.no_grad():
        for images, labels in loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            preds = torch.argmax(outputs.logits, dim=1)
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
    return accuracy_score(all_labels, all_preds)

# Perform evaluation on the test set
test_accuracy = evaluate(test_loader)

# Print results
print(f"Test Accuracy: {test_accuracy:.2%}")
print(f"Total Execution Time: {time.time() - start_time:.2f} seconds")





Resolving data files:   0%|          | 0/5400 [00:00<?, ?it/s]

Test Accuracy: 0.19%
Total Execution Time: 131.96 seconds
